#### AI모델이 대화를 처리할 때, 처리해야 할 정보의 양을 줄여서 더 빠르고 효율적으로 응답할 수 있도록 하는 방법

In [1]:
import sys

sys.path.append("C:/cloud_backend/langchain_app")

from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import trim_messages
from langchain_openai import ChatOpenAI
from memory_history import get_session_history
from operator import itemgetter
from dotenv import load_dotenv


load_dotenv("../../.env")

model = ChatOpenAI(model="gpt-4o-mini")



prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 금융 상담사입니다. 모든 질문에 최선을 다해 답변하십시오."),
        ("placeholder", "{chat_history}"),  # 이전 대화 이력
        ("human", "{input}"),  # 사용자의 새로운 질문
    ]
)


# 메시지 트리밍 유틸리티 설정( 가장 최근 2개의 메세지만 유지 )
trimmer = trim_messages(strategy="last", max_tokens=2, token_counter=len)

# 트리밍된 대화 이력과 함께 체인 실행
chain_with_trimming = (
    #RunnablePassthrough.assign()은 반드시 딕셔너리(dict) 입력만 받음
    RunnablePassthrough.assign(chat_history=itemgetter("chat_history") | trimmer)
    | prompt
    | model
)

# 트리밍된 대화 이력을 사용하는 체인 설정
chain_with_trimmed_history = RunnableWithMessageHistory(
    chain_with_trimming,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    # input_messages_key, history_messages_key 항목을 제공하지 않으면 메시지 리스트(list) 형태로 변환하여 반환됨
)

# 새로운 대화 내용 추가 후 체인 실행
answer = chain_with_trimmed_history.invoke(
    {"input": "저는 5년 내에 집을 사기 위해 어떤 재정 계획을 세워야 하나요?"},  # 사용자의 질문
    {"configurable": {"session_id": "user01"}}  # 세션 ID 설정
).content
print(answer)


c:\cloud_backend\langchain_app\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


5년 내에 집을 사기 위해서는 체계적인 재정 계획을 세우는 것이 중요합니다. 다음은 도움이 될 수 있는 몇 가지 단계입니다.

1. **목표 설정**:
   - 원하는 집의 가격 범위를 설정하세요. 지역에 따라 집값이 다르므로, 자신이 원하는 지역의 평균 주택 가격을 조사하세요.
   - 구체적인 예산을 세우고, 얼마나 대출을 받을 수 있을지를 고려해보세요.

2. **저축 계획**:
   - **다운 페이먼트**: 대부분의 주택 구매자는 가격의 10%에서 20%를 다운페이먼트로 지불합니다. 따라서 목표 금액을 계산하고 매달 얼마를 저축해야 할지 계획하세요.
   - **저축 계좌**: 고수익 저축 계좌나 정기예금 계좌를 활용해 안정적으로 저축하세요. 필요시 자동이체를 설정하여 정기적으로 저축할 수 있도록 하세요.

3. **예산 관리**:
   - 가계 예산을 세워 수입과 지출을 철저히 관리하세요. 불필요한 지출을 줄이고, 추가로 저축할 수 있는 여유 자금을 확보하세요.
   - 주기적으로 예산을 검토하고 필요하면 조정하세요.

4. **신용 점수 관리**:
   - 주택 담보 대출을 받기 위해서는 좋은 신용 점수가 필요합니다. 신용 점수를 확인하고, 높일 수 있는 방법에 대해 계획하세요. 예를 들어, 카드 대금을 제때 지급하고, 신용카드의 사용률을 낮추는 것이 좋습니다.

5. **대출 옵션 조사**:
   - 다양한 대출 상품을 비교 분석하여 가장 유리한 조건을 찾아보세요. 가상의 대출 상환 시나리오를 계산해보는 것도 도움이 됩니다.
   - 저금리 환경에서는 고정 금리 대출이 유리할 수 있습니다.

6. **전문가 상담**:
   - 필요하다면 금융 전문가나 주택 상담사와 상담하여 구체적인 계획과 전략을 세우는 것도 좋은 방법입니다. 

7. **비상 자금 마련**:
   - 주택 구매 비용 외에도 예기치 않은 상황에 대비해 비상 자금을 마련해 두는 것이 중요합니다. 보통 3~6개월의 생활비를 비상 자금으로 추천합니다.

이러한 단계들을 따라 체계적으로

In [2]:
answer = chain_with_trimmed_history.invoke(
    {"input": "내가 방금 뭐라고 했나요?"},  # 사용자의 질문
    {"configurable": {"session_id": "user01"}}
).content
print(answer)

당신은 "저는 5년 내에 집을 사기 위해 어떤 재정 계획을 세워야 하나요?"라고 질문하셨습니다. 저는 그에 대해 단계별 재정 계획을 제안했습니다. 추가적인 질문이 있으시면 언제든지 말씀해 주세요!
